In [3]:
import pandas as pd
import re

# Load the nginx log file
def parse_nginx_log(log_file):
    logs = []
    regex = r"^(\S+) - - \[(.*?)\] \"(\S+) (.*?) (HTTP\/\d\.\d)\" (\d{3}) (\d+) \"(.*?)\" \"(.*?)\""
    
    with open(log_file, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            match = re.match(regex, line)
            if match:
                logs.append(match.groups())

    columns = ["client_ip", "timestamp", "method", "request", "protocol", "status", "response_size", "referrer", "user_agent"]
    return pd.DataFrame(logs, columns=columns)

# Labeling rules
def label_data(df):
    labels = []
    
    for _, row in df.iterrows():
        if int(row['status']) == 404 and row['request'].endswith('/wp-login.php'):
            labels.append("Brute Force")
        elif re.search(r".*(UNION|SELECT|INSERT|DELETE|UPDATE).*", row['request'], re.IGNORECASE):
            labels.append("SQL Injection")
        elif re.search(r".*(<script>|%3Cscript%3E).*", row['request'], re.IGNORECASE):
            labels.append("XSS Attack")
        elif '../' in row['request'] or '..%2F' in row['request']:
            labels.append("Directory Traversal")
        else:
            labels.append("Normal")
    
    df['label'] = labels
    return df

# Main pipeline
log_file = './dataset/dataset.txt'  # Path to your dataset file

try:
    # Step 1: Parse Logs
    raw_data = parse_nginx_log(log_file)
    print("Log parsing successful.")
    print(raw_data.head())
    
    # Step 2: Label Data
    labeled_data = label_data(raw_data)
    print("Labeling completed.")
    print(labeled_data.head())
    
    # Save to CSV
    output_file = './dataset/labeled_logs.csv'
    labeled_data.to_csv(output_file, index=False)
    print(f"Labeled data saved to {output_file}")
except Exception as e:
    print(f"An error occurred: {e}")


Log parsing successful.
       client_ip                   timestamp method  \
0  10.220.34.199  11/Apr/2023:18:41:17 +0700   POST   
1  52.167.144.35  11/Apr/2023:18:41:17 +0700    GET   
2   36.69.15.218  11/Apr/2023:18:41:21 +0700    GET   
3   36.69.15.218  11/Apr/2023:18:41:26 +0700    GET   
4  10.220.34.199  11/Apr/2023:18:41:28 +0700   POST   

                                             request  protocol status  \
0  /wp-cron.php?doing_wp_cron=1681213277.42503809...  HTTP/1.0    200   
1                                  /page-sitemap.xml  HTTP/1.0    200   
2                                                  /  HTTP/1.0    301   
3                                                  /  HTTP/1.0    301   
4  /wp-cron.php?doing_wp_cron=1681213288.58341407...  HTTP/1.0    200   

  response_size referrer                                         user_agent  
0             0        -                                       XDGTJnG3Fvej  
1         36653        -  Mozilla/5.0 AppleWebKit/